# Corporate Bankruptcy Prediction: full code walkthrough

UCI Polish Companies Bankruptcy (id 365). This notebook contains **all the project code** from `src/` in its cells, with a short explanation above each piece, and trains the five models **one at a time**. It produces the same results as `python src/train.py`.

**How to use:** start Jupyter from the project folder, open this notebook from `notebooks/`, and use Kernel → Restart & Run All. It needs internet for the UCI fetch, takes several minutes, and **overwrites** the files in `results/` and `models/`.

**Order:** 1) code sections (run once, nothing visible happens), 2) the analysis steps, 3) one cell per model.

# Part 1: The code

## Settings and helpers (`utils.py`)

Paths, random seed and settings shared by every step: seed 42, 80/20 split, 5 CV folds, up to 6 search candidates per model, and PR-AUC as the tuning metric. In a notebook, the project root is the folder above `notebooks/`.

In [ ]:
import json
import os
import random
import re
import sys
from pathlib import Path
import numpy as np
import matplotlib
IN_NOTEBOOK = "ipykernel" in sys.modules
if not IN_NOTEBOOK:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"
MODELS_DIR = ROOT / "models"
RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5
N_ITER = 6
TUNING_METRIC = "average_precision"
CLASS_NAMES = {0: "Non-Bankrupt", 1: "Bankrupt"}
POSITIVE_CLASS = 1

**`set_seed`**: Fix the Python and NumPy random seeds so reruns give the same results.

In [ ]:
def set_seed(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

**`ensure_dirs`**: Create the figures, metrics and models folders if they do not exist.

In [ ]:
def ensure_dirs() -> None:
    for d in (FIGURES_DIR, METRICS_DIR, MODELS_DIR):
        d.mkdir(parents=True, exist_ok=True)

**`slugify`**: Turn a model name into a file-name-friendly string, for example `Random Forest` becomes `random_forest`.

In [ ]:
def slugify(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")

**`save_fig`**: Save a figure to `results/figures`, show it in the notebook and close it.

In [ ]:
def save_fig(fig, filename: str) -> Path:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=150, bbox_inches="tight")
    if IN_NOTEBOOK:
        plt.show()
    plt.close(fig)
    print(f"  [saved figure] {path.relative_to(ROOT)}")
    return path

**`save_json`**: Write a dictionary to a JSON file.

In [ ]:
def save_json(obj, path: Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    print(f"  [saved] {Path(path).relative_to(ROOT)}")

**`section`**: Print a banner heading for each step.

In [ ]:
def section(title: str) -> None:
    print("\n" + "=" * 78 + f"\n{title}\n" + "=" * 78)

## Loading, cleaning and inspecting the data (`data_loader.py`)

The data is fetched from UCI at run time with `ucimlrepo`. Nothing is stored in the repo.

In [ ]:
import numpy as np
import pandas as pd

**`load_data`**: Fetch the UCI Polish Companies Bankruptcy dataset (id=365) -- no manual CSV.

In [ ]:
def load_data():
    from ucimlrepo import fetch_ucirepo

    polish_companies_bankruptcy = fetch_ucirepo(id=365)
    X = polish_companies_bankruptcy.data.features
    y = polish_companies_bankruptcy.data.targets

    md = polish_companies_bankruptcy.metadata
    print(f"Source: UCI Machine Learning Repository (id=365) | name: {md.get('name')}")
    print(f"        {md.get('repository_url')}")

    if y.shape[1] != 1:
        raise ValueError(f"Expected one target column, got {list(y.columns)}")
    target_name = y.columns[0]
    y = pd.to_numeric(y.iloc[:, 0], errors="raise")
    if y.isna().any():
        raise ValueError("Target contains missing values.")
    if not set(y.unique()) <= {0, 1}:
        raise ValueError(f"Target labels must be {{0,1}} (1 = bankrupt); found {sorted(y.unique())}")
    y = y.astype(int).reset_index(drop=True)

    raw_dtypes = X.dtypes.astype(str)
    X_num = X.apply(pd.to_numeric, errors="coerce").reset_index(drop=True)
    coerced = int(X_num.isna().sum().sum() - X.isna().sum().sum())
    if coerced:
        print(f"WARNING: {coerced} non-numeric feature values were coerced to NaN.")

    info = {"target_name": target_name, "raw_dtypes": raw_dtypes, "coerced_to_nan": coerced,
            "source": "UCI id=365"}
    return X_num, y, info

**`clean_data`**: Drop exact duplicate rows (features + target) so a row cannot land in both train and test.

In [ ]:
def clean_data(X, y):
    section("DATA CLEANING")
    dup = pd.concat([X, y.rename("__target__")], axis=1).duplicated(keep="first").values
    n_dup = int(dup.sum())
    print(f"Exact duplicate rows (features + target) removed: {n_dup}")
    X, y = X[~dup].reset_index(drop=True), y[~dup].reset_index(drop=True)
    print(f"Shape after cleaning: {X.shape}")
    return X, y, n_dup

**`inspect_dataset`**: Step 1 report on the raw data: size, class counts, missing and infinite values, duplicates, constant and near-constant features, and outliers (IQR rule, analysed only, never removed). Saves `feature_summary.csv` and `dataset_inspection.json`.

In [ ]:
def inspect_dataset(X, y, info):
    section("STEP 1 — DATASET INSPECTION")
    n_rows, n_feat = X.shape
    raw = info["raw_dtypes"]
    report = {"rows": n_rows, "features": n_feat}

    print(f"Rows: {n_rows:,}")
    print(f"Feature columns: {n_feat} (+ target '{info['target_name']}' = {n_feat + 1} columns total)")
    print("\nFeature names and raw dtypes:\n  " + ", ".join(f"{c}:{raw[c]}" for c in X.columns))
    print(f"Dtype summary: {raw.value_counts().to_dict()}")

    counts = y.value_counts().sort_index()
    print(f"\nTarget column: {info['target_name']}")
    for lab, c in counts.items():
        print(f"  class {lab} ({CLASS_NAMES[lab]}): {c:,} ({c / n_rows * 100:.2f}%)")
    report["class_counts"] = {CLASS_NAMES[k]: int(v) for k, v in counts.items()}

    miss = X.isna().sum()
    total_missing = int(miss.sum())
    print(f"\nTotal missing values: {total_missing:,} "
          f"({total_missing / X.size * 100:.2f}% of cells); "
          f"rows with >=1 missing: {int(X.isna().any(axis=1).sum()):,}")
    print(f"Features with any missing: {int((miss > 0).sum())}/{n_feat}")
    top_miss = miss[miss > 0].sort_values(ascending=False).head(15)
    if len(top_miss):
        print("Top features by missing count (full table in feature_summary.csv):")
        print(top_miss.to_frame("n_missing").assign(pct=lambda d: (d.n_missing / n_rows * 100).round(2)).to_string())
    arr = X.to_numpy(dtype=float)
    n_inf = int(np.isinf(arr).sum())
    print(f"Infinite values: {n_inf}")
    report.update(total_missing=total_missing, infinite_values=n_inf)

    dup_feat = int(X.duplicated().sum())
    dup_full = int(pd.concat([X, y.rename("__t__")], axis=1).duplicated().sum())
    print(f"\nDuplicate rows: {dup_full} (features+target), {dup_feat} (features only)")
    report.update(duplicate_rows_full=dup_full, duplicate_rows_features_only=dup_feat)

    nunique = X.nunique(dropna=True)
    top_share = X.apply(lambda s: s.value_counts(normalize=True).iloc[0] if s.notna().any() else np.nan)
    constant = list(nunique[nunique <= 1].index)
    near_const = list(top_share[(top_share >= 0.99) & (nunique > 1)].index)
    print(f"Constant features: {constant if constant else 'none'}")
    print(f"Near-constant features (>=99% one value): {near_const if near_const else 'none'}")
    report.update(constant_features=constant, near_constant_features=near_const)

    q1, q3 = X.quantile(0.25), X.quantile(0.75)
    iqr = q3 - q1
    ok = iqr[iqr > 0].index
    mask = X[ok].lt(q1[ok] - 1.5 * iqr[ok], axis=1) | X[ok].gt(q3[ok] + 1.5 * iqr[ok], axis=1)
    out_frac = mask.mean()
    per_class = mask.sum(axis=1).groupby(y.values).mean()
    print("\nOUTLIER ANALYSIS (1.5*IQR rule; analysis only, nothing removed)")
    print(f"  Median share of values flagged per feature: {out_frac.median() * 100:.2f}%")
    print(f"  Features with >10% flagged values: {int((out_frac > 0.10).sum())}/{len(ok)}")
    print("  Top 10 features by flagged share (%):")
    print((out_frac.sort_values(ascending=False).head(10) * 100).round(2).to_string())
    print(f"  Largest |value| in any feature: {np.abs(arr[np.isfinite(arr)]).max():.4g}")
    print("  Mean number of outlier-flagged features per row, by class:")
    for lab, v in per_class.items():
        print(f"    {CLASS_NAMES[lab]}: {v:.2f}")
    if per_class.get(1, 0) > per_class.get(0, 0):
        print("  Finding: bankrupt firms are flagged in MORE features per row, so extremes carry class signal.")
    else:
        print("  Finding: bankrupt firms do not show a higher outlier load in this data.")
    print("  Decision: outliers are KEPT (financial ratios are heavy-tailed). Their influence is handled in "
          "the pipeline via median imputation, RobustScaler + fixed clipping (scale-sensitive models) and "
          "tree-based models.")
    report["outlier_median_share"] = float(out_frac.median())
    report["outlier_features_per_row_by_class"] = {CLASS_NAMES[k]: float(v) for k, v in per_class.items()}

    table = pd.DataFrame({
        "raw_dtype": raw, "n_missing": miss, "pct_missing": miss / n_rows * 100,
        "n_unique": nunique, "top_value_share": top_share,
        "outlier_fraction_iqr": out_frac.reindex(X.columns),
        "skewness": X.skew(), "min": X.min(), "max": X.max()})
    table.to_csv(METRICS_DIR / "feature_summary.csv")
    save_json(report, METRICS_DIR / "dataset_inspection.json")
    return report

## Exploratory data analysis (`eda.py`)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

**`run_eda`**: Step 2, descriptive only: class distribution, summary statistics, missing-value chart, Spearman correlation heatmap with the top correlated pairs, and histograms of the features most associated with the target.

In [ ]:
def run_eda(X, y):
    section("STEP 2 — EXPLORATORY DATA ANALYSIS")
    n = len(y)

    counts = y.value_counts().sort_index()
    tgt = pd.DataFrame({"class": [CLASS_NAMES[i] for i in counts.index], "count": counts.values,
                        "percent": (counts.values / n * 100).round(3)})
    tgt.to_csv(METRICS_DIR / "class_distribution.csv", index=False)
    print(tgt.to_string(index=False))
    fig, ax = plt.subplots(figsize=(5, 4))
    bars = ax.bar(tgt["class"], tgt["count"], color=["#4c72b0", "#c44e52"])
    for b, c, p in zip(bars, tgt["count"], tgt["percent"]):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{c:,}\n({p:.2f}%)", ha="center", va="bottom")
    ax.set_title("Class distribution"); ax.set_ylabel("Companies")
    ax.set_ylim(0, tgt["count"].max() * 1.18)
    save_fig(fig, "class_distribution.png")

    summary = X.describe().T
    summary["median"] = X.median()
    summary["skew"] = X.skew()
    summary.to_csv(METRICS_DIR / "summary_statistics.csv")
    print(f"\nSummary statistics saved ({summary.shape[0]} features). Most skewed features (|skew|):")
    print(summary["skew"].abs().sort_values(ascending=False).head(5).round(2).to_string())

    miss_pct = (X.isna().mean() * 100).sort_values(ascending=False)
    if (miss_pct > 0).any():
        top = miss_pct[miss_pct > 0].head(25)
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.barh(top.index[::-1], top.values[::-1], color="#dd8452")
        ax.set_xlabel("% missing"); ax.set_title("Missing values (top features)")
        save_fig(fig, "missing_values.png")
    else:
        print("No missing values -> missing-value figure skipped.")

    corr = X.corr(method="spearman")
    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(np.ma.masked_invalid(corr.values), cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr))); ax.set_yticks(range(len(corr)))
    ax.set_xticklabels(corr.columns, rotation=90, fontsize=5)
    ax.set_yticklabels(corr.columns, fontsize=5)
    fig.colorbar(im, ax=ax, shrink=0.8, label="Spearman rho")
    ax.set_title("Feature correlation heatmap (Spearman)")
    save_fig(fig, "correlation_heatmap.png")
    pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
    pairs = pairs.reindex(pairs.abs().sort_values(ascending=False).index)
    pairs.head(30).rename("spearman_rho").to_csv(METRICS_DIR / "top_correlated_pairs.csv")
    print(f"\nFeature pairs with |rho| >= 0.9: {int((pairs.abs() >= 0.9).sum())} "
          f"(top pairs in top_correlated_pairs.csv)")

    assoc = X.corrwith(y, method="spearman").abs().sort_values(ascending=False).head(12)
    fig, axes = plt.subplots(4, 3, figsize=(13, 12))
    for ax, col in zip(axes.ravel(), assoc.index):
        lo, hi = X[col].quantile([0.01, 0.99])
        for lab, color in ((0, "#4c72b0"), (1, "#c44e52")):
            vals = X.loc[y == lab, col].dropna().clip(lo, hi)
            ax.hist(vals, bins=40, range=(lo, hi), density=True, alpha=0.55, color=color, label=CLASS_NAMES[lab])
        ax.set_title(f"{col} (|rho| with target = {assoc[col]:.2f})", fontsize=9)
    axes[0, 0].legend(fontsize=8)
    fig.suptitle("Feature distributions by class (clipped to 1st-99th percentile for display)")
    fig.tight_layout()
    save_fig(fig, "feature_distributions.png")

## Preprocessing pipeline and imbalance analysis (`preprocessing.py`)

In [ ]:
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, RobustScaler
CLIP_VALUE = 10.0

**`_inf_to_nan`**: Replace infinite values (from division by zero) with NaN so they can be imputed.

In [ ]:
def _inf_to_nan(X):
    X = np.array(X, dtype=float)
    X[~np.isfinite(X)] = np.nan
    return X

**`_clip`**: Clip values to the range of plus or minus `CLIP_VALUE` (10). Used for the linear models only.

In [ ]:
def _clip(X):
    return np.clip(X, -CLIP_VALUE, CLIP_VALUE)

**`build_preprocessor`**: Leakage-safe preprocessing; every learned step is fitted on training data only

```text
(the pipeline is refit inside each CV fold).

inf -> NaN      : infinities (ratios with zero denominators) cannot be imputed/scaled.
median impute   : financial ratios are heavy-tailed, the median is robust; the dataset has missing values.
variance filter : drops features that are constant in the training data (no information).
RobustScaler    : (scale=True) median/IQR scaling, robust to outliers; needed by LR / linear SVM.
clip +-10       : (scale=True) bounds extreme robust-scaled values so a few points cannot dominate
                  the linear models. Not applied to tree models (scale-invariant, split on ranks).
```

In [ ]:
def build_preprocessor(scale: bool) -> Pipeline:
    steps = [
        ("inf_to_nan", FunctionTransformer(_inf_to_nan, feature_names_out="one-to-one")),
        ("imputer", SimpleImputer(strategy="median")),
        ("variance", VarianceThreshold(threshold=0.0)),
    ]
    if scale:
        steps += [("scaler", RobustScaler()),
                  ("clip", FunctionTransformer(_clip, feature_names_out="one-to-one"))]
    return Pipeline(steps)

**`analyze_imbalance`**: Step 3: print the class ratio, show why always predicting healthy is misleading, compare undersampling, oversampling, SMOTE and class weights, and return the facts as a dictionary.

In [ ]:
def analyze_imbalance(y):
    section("STEP 3 — CLASS IMBALANCE ANALYSIS")
    counts = y.value_counts()
    minority, majority = counts.idxmin(), counts.idxmax()
    n_min, n_maj, n = int(counts.min()), int(counts.max()), len(y)
    ratio = n_maj / n_min
    print(f"Minority: {CLASS_NAMES[minority]} = {n_min:,} | Majority: {CLASS_NAMES[majority]} = {n_maj:,}")
    print(f"Imbalance ratio (majority:minority) = {ratio:.1f}:1")
    print(f"A trivial 'always {CLASS_NAMES[majority]}' classifier would reach {n_maj / n * 100:.2f}% accuracy "
          "-> accuracy is misleading; PR-AUC / recall / F1 are emphasised.\n")

    print("Options evaluated against these numbers:")
    print(f"  class weights        : no data change; reweights loss by ~{ratio:.1f}x for the minority class.")
    print(f"  random undersampling : would discard {n_maj - n_min:,} majority rows ({(n_maj - n_min) / n * 100:.1f}% of data).")
    print(f"  random oversampling  : would duplicate minority rows ~{ratio:.1f}x -> overfitting risk.")
    print(f"  SMOTE                : would synthesise ~{n_maj - n_min:,} points by interpolation in a heavy-tailed, "
          "highly correlated ratio space; costly and can create unrealistic firms.")
    strategy = "class_weight"
    print(f"\nCHOSEN: {strategy}. The minority class has {n_min:,} real examples, so reweighting is sufficient, "
          "keeps all real data, adds no synthetic rows, and cannot leak into the test set.")
    if n_min < 500:
        print("NOTE: minority class is small (<500); consider SMOTE inside an imblearn Pipeline "
              "(training folds only) -- not implemented here.")
    return {"minority_class": CLASS_NAMES[minority], "minority_count": n_min, "majority_count": n_maj,
            "imbalance_ratio": float(ratio), "strategy": strategy}

**Pickle compatibility.** The saved model refers to the two helper functions above by module name. These lines register them under the name `preprocessing`, so the model saved from this notebook can also be loaded by `python src/predict.py`.

In [ ]:
import types

preprocessing = types.ModuleType("preprocessing")
sys.modules["preprocessing"] = preprocessing
for _f in (_inf_to_nan, _clip):
    _f.__module__ = "preprocessing"
    setattr(preprocessing, _f.__name__, _f)

## Models, split and saving (`train.py`)

The five models and their search spaces, the train/test split and the saving helpers. The tuning loop is shown model by model further down.

In [ ]:
import argparse
import platform
import time
import joblib
import pandas as pd
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterGrid, RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
SCORING = {"average_precision": "average_precision", "roc_auc": "roc_auc", "f1": "f1"}

**`get_model_specs`**: Models, why they are included, and (deliberately small) search spaces. All use class weighting.

In [ ]:
def get_model_specs():
    rs = RANDOM_STATE
    return {
        "Logistic Regression": dict(
            estimator=LogisticRegression(class_weight="balanced", max_iter=2000, random_state=rs),
            scale=True, grid={"clf__C": [0.01, 0.1, 1, 10]},
            why="Interpretable linear baseline; coefficients show direction/size of effects."),
        "Decision Tree": dict(
            estimator=DecisionTreeClassifier(class_weight="balanced", random_state=rs),
            scale=False, grid={"clf__max_depth": [4, 6, 8, 12], "clf__min_samples_leaf": [5, 20, 50]},
            why="Transparent rule-based model; captures non-linearity; depth/leaf limits control overfitting."),
        "Random Forest": dict(
            estimator=RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample", random_state=rs),
            scale=False, grid={"clf__max_depth": [8, 16, None], "clf__min_samples_leaf": [1, 5, 10],
                               "clf__max_features": ["sqrt", 0.3]},
            why="Bagged trees: reduces tree variance, handles interactions, gives feature importances."),
        "Linear SVM": dict(
            estimator=LinearSVC(class_weight="balanced", dual=False, max_iter=5000, random_state=rs),
            scale=True, grid={"clf__C": [0.01, 0.1, 1]},
            why="Margin-based classifier. LinearSVC (not kernel SVC) is used because kernel SVC training "
                "scales poorly with tens of thousands of rows. Provides margins, not probabilities."),
        "Gradient Boosting": dict(
            estimator=HistGradientBoostingClassifier(class_weight="balanced", early_stopping=False, random_state=rs),
            scale=False, grid={"clf__learning_rate": [0.03, 0.1], "clf__max_iter": [200, 400],
                               "clf__max_leaf_nodes": [15, 31], "clf__l2_regularization": [0.0, 1.0]},
            why="Boosted trees (sklearn histogram-based implementation, chosen over GradientBoostingClassifier "
                "for runtime on tens of thousands of rows); typically strong on tabular data."),
    }

**`split_data`**: Step 4: stratified 80/20 split. The test set is not touched until the final evaluation.

In [ ]:
def split_data(X, y):
    section("STEP 4 — STRATIFIED TRAIN / TEST SPLIT")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
    for nm, yy in (("train", y_train), ("test", y_test)):
        c = yy.value_counts().sort_index()
        print(f"{nm}: {len(yy):,} rows | " + ", ".join(f"{CLASS_NAMES[k]}={v:,} ({v / len(yy) * 100:.2f}%)" for k, v in c.items()))
    print("Test set is not touched until the final evaluation. Imputation/scaling/etc. are fitted inside "
          "pipelines on training data only (refit per CV fold).")
    return X_train, X_test, y_train, y_test

**`make_cv`**: Stratified k-fold splitter used while tuning.

In [ ]:
def make_cv(n_splits=CV_FOLDS):
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

**`save_final_model`**: Step 11: save the best pipeline plus metadata (feature names, best parameters, test metrics, library versions).

In [ ]:
def save_final_model(best_name, models, searches, comparison, X_train):
    section("STEP 11 — SAVE FINAL MODEL")
    pipe = models[best_name]
    path = MODELS_DIR / "best_model.joblib"
    joblib.dump(pipe, path)
    meta = {
        "model_name": best_name,
        "selection_rule": "highest cross-validated PR-AUC on training data",
        "feature_names": list(X_train.columns),
        "class_names": {str(k): v for k, v in CLASS_NAMES.items()},
        "has_predict_proba": hasattr(pipe, "predict_proba"),
        "best_params": searches[best_name].best_params_,
        "test_metrics": comparison.loc[best_name].to_dict(),
        "random_state": RANDOM_STATE,
        "sklearn_version": sklearn.__version__, "python_version": platform.python_version(),
    }
    save_json(meta, MODELS_DIR / "model_metadata.json")
    print(f"  [saved model] {path.relative_to(ROOT)}  ({best_name})")
    return path

**`save_sample_input`**: Save 5 held-out test rows, with their true class, for trying `predict.py`.

In [ ]:
def save_sample_input(X_test, y_test, n=5):
    out = X_test.head(n).copy()
    out["true_class"] = y_test.loc[out.index].values
    p = ROOT / "results" / "sample_input.csv"
    out.to_csv(p, index=False)
    print(f"  [saved] {p.relative_to(ROOT)} ({n} held-out test rows for trying predict.py)")

**`tune_model`**: tunes one model with `RandomizedSearchCV` (stratified k-fold on training data only, scored by PR-AUC, refit on PR-AUC) and saves its `cv_results_*.csv`. This is the body of the loop in `train.py`, run for one model at a time.

In [ ]:
def tune_model(name, spec, X_train, y_train, cv, n_iter=N_ITER):
    print(f"\n>>> {name}\n    why: {spec['why']}")
    pipe = Pipeline([("prep", build_preprocessor(spec["scale"])), ("clf", spec["estimator"])])
    n_cand = min(n_iter, len(ParameterGrid(spec["grid"])))
    search = RandomizedSearchCV(pipe, spec["grid"], n_iter=n_cand, scoring=SCORING, refit=TUNING_METRIC,
                                cv=cv, n_jobs=-1, random_state=RANDOM_STATE, error_score="raise")
    t0 = time.time()
    search.fit(X_train, y_train)
    i = search.best_index_
    print(f"    best params: {search.best_params_}")
    print(f"    CV PR-AUC={search.cv_results_['mean_test_average_precision'][i]:.4f} "
          f"ROC-AUC={search.cv_results_['mean_test_roc_auc'][i]:.4f} "
          f"F1={search.cv_results_['mean_test_f1'][i]:.4f} | {time.time() - t0:.0f}s")
    pd.DataFrame(search.cv_results_).to_csv(METRICS_DIR / f"cv_results_{slugify(name)}.csv", index=False)
    return search

**`save_hyperparameters`**: save the fixed settings, search space and best parameters of every model to `hyperparameters.json`.

In [ ]:
def save_hyperparameters(searches, specs):
    save_json({n: {"fixed_hyperparameters": specs[n]["estimator"].get_params(deep=False),
                   "search_space": specs[n]["grid"], "scaled": specs[n]["scale"],
                   "best_params": s.best_params_, "why": specs[n]["why"]} for n, s in searches.items()},
              METRICS_DIR / "hyperparameters.json")

## Evaluation, comparison and interpretation (`evaluate.py`)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, average_precision_score,
                             classification_report, confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.tree import plot_tree
LABELS = [CLASS_NAMES[0], CLASS_NAMES[1]]

**`get_scores`**: Probability of the positive (Bankrupt) class, or the margin if no probabilities exist.

In [ ]:
def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)

**`compute_metrics`**: Accuracy, precision, recall and F1 at the model's default decision rule, plus ROC-AUC and PR-AUC computed from the scores.

In [ ]:
def compute_metrics(y_true, y_pred, scores):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, scores),
        "PR-AUC": average_precision_score(y_true, scores),
    }

**`evaluate_models`**: Evaluate fitted pipelines on the untouched test set. Returns (metrics DataFrame, scores dict).

In [ ]:
def evaluate_models(models, X_test, y_test):
    section("STEP 8 — EVALUATION (held-out test set)")
    rows, all_scores = {}, {}
    for name, model in models.items():
        y_pred = model.predict(X_test)
        scores = get_scores(model, X_test)
        all_scores[name] = scores
        rows[name] = compute_metrics(y_test, y_pred, scores)
        slug = slugify(name)

        report = classification_report(y_test, y_pred, target_names=LABELS, zero_division=0)
        (METRICS_DIR / f"classification_report_{slug}.txt").write_text(report)
        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        pd.DataFrame(cm, index=[f"True {l}" for l in LABELS], columns=[f"Pred {l}" for l in LABELS]) \
            .to_csv(METRICS_DIR / f"confusion_matrix_{slug}.csv")
        print(f"\n--- {name} ---\n{report}Confusion matrix [rows=true, cols=pred] (order: {LABELS}):\n{cm}")

        fig, ax = plt.subplots(figsize=(4.5, 4))
        ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=LABELS, ax=ax,
                                                colorbar=False, values_format="d")
        ax.set_title(f"Confusion matrix — {name}")
        save_fig(fig, f"confusion_matrix_{slug}.png")

    fig, ax = plt.subplots(figsize=(6, 5))
    for name, s in all_scores.items():
        fpr, tpr, _ = roc_curve(y_test, s)
        ax.plot(fpr, tpr, label=f"{name} (AUC={rows[name]['ROC-AUC']:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="Chance")
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title("ROC curves (test set)"); ax.legend(fontsize=8)
    save_fig(fig, "roc_curves.png")

    fig, ax = plt.subplots(figsize=(6, 5))
    for name, s in all_scores.items():
        p, r, _ = precision_recall_curve(y_test, s)
        ax.plot(r, p, label=f"{name} (AP={rows[name]['PR-AUC']:.3f})")
    ax.axhline(float(np.mean(y_test)), color="k", ls="--", lw=0.8, label="No-skill (prevalence)")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall curves (test set)"); ax.legend(fontsize=8)
    save_fig(fig, "pr_curves.png")

    return pd.DataFrame(rows).T, all_scores

**`build_comparison`**: Comparison table. Models are RANKED by cross-validated PR-AUC on the training data only, so the

```text
test set plays no role in model selection; test metrics are reported alongside.
```

In [ ]:
def build_comparison(test_df, searches):
    section("STEP 9 — MODEL COMPARISON")
    cv = {}
    for name, s in searches.items():
        i = s.best_index_
        cv[name] = {"CV PR-AUC": s.cv_results_["mean_test_average_precision"][i],
                    "CV ROC-AUC": s.cv_results_["mean_test_roc_auc"][i],
                    "CV F1": s.cv_results_["mean_test_f1"][i]}
    table = test_df.join(pd.DataFrame(cv).T)
    table["Rank (CV PR-AUC)"] = table["CV PR-AUC"].rank(ascending=False, method="min").astype(int)
    table = table.sort_values("Rank (CV PR-AUC)")
    table.index.name = "Model"
    table.to_csv(METRICS_DIR / "model_comparison.csv")
    print("Test-set metrics + cross-validation metrics (ranked by CV PR-AUC, training data only):")
    print(table.round(4).to_string())
    print(f"\nSelected model (best CV PR-AUC): {table.index[0]}")
    return table

**`transformed_feature_names`**: Feature names after preprocessing (constant columns are removed), so importances line up with the right columns.

In [ ]:
def transformed_feature_names(pipe, input_cols):
    try:
        return list(pipe[:-1].get_feature_names_out(np.asarray(input_cols, dtype=object)))
    except Exception:
        n = pipe[:-1].transform(pd.DataFrame(columns=input_cols, data=np.zeros((1, len(input_cols))))).shape[1]
        return [f"f{i}" for i in range(n)]

**`_barh`**: Helper that draws and saves a horizontal bar chart.

In [ ]:
def _barh(series, title, xlabel, filename, xerr=None):
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.barh(series.index[::-1], series.values[::-1], xerr=None if xerr is None else xerr[::-1], color="#4c72b0")
    ax.set_title(title); ax.set_xlabel(xlabel)
    save_fig(fig, filename)

**`interpret_models`**: Step 10: logistic regression coefficients, decision-tree plot (top 3 levels), Random Forest importances, and permutation importance for the selected model.

In [ ]:
def interpret_models(models, best_name, X_test, y_test):
    section("STEP 10 — MODEL INTERPRETATION")
    cols = list(X_test.columns)

    if "Logistic Regression" in models:
        pipe = models["Logistic Regression"]
        coef = pd.Series(pipe.named_steps["clf"].coef_[0], index=transformed_feature_names(pipe, cols))
        coef.sort_values(key=np.abs, ascending=False).to_csv(METRICS_DIR / "logistic_regression_coefficients.csv",
                                                             header=["coefficient"])
        top = coef.reindex(coef.abs().sort_values(ascending=False).head(20).index)
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.barh(top.index[::-1], top.values[::-1], color=["#c44e52" if v > 0 else "#4c72b0" for v in top.values[::-1]])
        ax.set_title("Logistic Regression — top 20 coefficients (scaled features)\nred: raises Bankrupt odds, blue: lowers")
        ax.set_xlabel("Coefficient")
        save_fig(fig, "interpret_logistic_coefficients.png")

    if "Decision Tree" in models:
        pipe = models["Decision Tree"]
        tree = pipe.named_steps["clf"]
        fig, ax = plt.subplots(figsize=(24, 10))
        plot_tree(tree, max_depth=3, feature_names=transformed_feature_names(pipe, cols), class_names=LABELS,
                  filled=True, rounded=True, fontsize=7, proportion=True, ax=ax)
        ax.set_title(f"Decision Tree — top 3 levels shown (fitted tree depth = {tree.get_depth()})")
        save_fig(fig, "interpret_decision_tree.png")

    if "Random Forest" in models:
        pipe = models["Random Forest"]
        imp = pd.Series(pipe.named_steps["clf"].feature_importances_, index=transformed_feature_names(pipe, cols))
        imp = imp.sort_values(ascending=False)
        imp.to_csv(METRICS_DIR / "random_forest_feature_importance.csv", header=["importance"])
        _barh(imp.head(20), "Random Forest — top 20 impurity-based importances", "Mean decrease in impurity",
              "interpret_random_forest_importance.png")

    print(f"Computing permutation importance for the selected model ({best_name}) ...")
    r = permutation_importance(models[best_name], X_test, y_test, scoring="average_precision",
                               n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
    pi = pd.DataFrame({"importance_mean": r.importances_mean, "importance_std": r.importances_std}, index=cols)
    pi = pi.sort_values("importance_mean", ascending=False)
    pi.to_csv(METRICS_DIR / "permutation_importance_best_model.csv")
    top = pi.head(20)
    _barh(top["importance_mean"], f"Permutation importance — {best_name}\n(drop in PR-AUC when feature is shuffled)",
          "Mean decrease in PR-AUC", "interpret_permutation_importance.png", xerr=top["importance_std"].values)
    print("Top 10 features by permutation importance:")
    print(pi.head(10).round(5).to_string())

## Prediction helpers (`predict.py`)

In [ ]:
import argparse
import json
import sys
from pathlib import Path
import joblib
import pandas as pd

**`load_artifacts`**: Load the saved model and its metadata.

In [ ]:
def load_artifacts():
    model_path, meta_path = MODELS_DIR / "best_model.joblib", MODELS_DIR / "model_metadata.json"
    if not model_path.exists() or not meta_path.exists():
        sys.exit("Saved model not found. Run `python src/train.py` first.")
    return joblib.load(model_path), json.loads(meta_path.read_text())

**`predict_frame`**: Check that all feature columns are present, convert them to numbers and predict (label, plus the bankrupt probability when the model has one).

In [ ]:
def predict_frame(df, model, meta):
    cols = meta["feature_names"]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required feature columns ({len(missing)}): {missing[:10]}"
                         f"{' ...' if len(missing) > 10 else ''}")
    X = df[cols].apply(pd.to_numeric, errors="coerce")
    pred = model.predict(X)
    proba = None
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)[:, list(model.classes_).index(1)]
    return pred, proba

**`format_prediction`**: Format one prediction as printable text.

In [ ]:
def format_prediction(pred, proba, meta, row=None):
    names = {int(k): v for k, v in meta["class_names"].items()}
    lines = [f"--- Row {row} ---"] if row is not None else []
    lines.append(f"Prediction: {names[int(pred)]}")
    if proba is not None:
        lines += [f"Bankrupt Probability: {proba * 100:.1f}%", f"Non-Bankrupt Probability: {(1 - proba) * 100:.1f}%"]
    else:
        lines.append(f"(Model '{meta['model_name']}' does not output probabilities.)")
    return "\n".join(lines)

# Part 2: Running the analysis

In [ ]:
set_seed()
ensure_dirs()

## Step 1: Load directly from UCI and inspect

In [ ]:
X, y, info = load_data()
report = inspect_dataset(X, y, info)

## Data cleaning and Step 2: EDA

In [ ]:
X, y, n_dup = clean_data(X, y)
run_eda(X, y)

## Step 3: Class imbalance

In [ ]:
imbalance = analyze_imbalance(y)
save_json(imbalance, METRICS_DIR / "imbalance_analysis.json")

## Step 4: Stratified train/test split

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y)

## Steps 5-7: Pipelines and tuning, one model at a time

Each model gets the same preprocessing pipeline (with scaling only for the linear models), the same stratified CV and the same scoring. Training data only; this may take several minutes in total.

In [ ]:
section("STEPS 5-7 — PREPROCESSING PIPELINES + HYPERPARAMETER TUNING (training data only)")
cv = make_cv(CV_FOLDS)
specs = get_model_specs()
searches = {}
print(f"RandomizedSearchCV | {cv.get_n_splits()}-fold stratified CV | refit/select on '{TUNING_METRIC}' "
      f"(PR-AUC) | max {N_ITER} candidates per model")

### Model 1: Logistic Regression

A weighted sum of the ratios turned into a probability, with one straight decision boundary. Interpretable baseline. Uses RobustScaler and clipping.

In [ ]:
name = "Logistic Regression"
print("search space:", specs[name]["grid"])
searches[name] = tune_model(name, specs[name], X_train, y_train, cv, N_ITER)

### Model 2: Decision Tree

A flowchart of if/else splits on ratio thresholds. Depth and leaf size limit overfitting.

In [ ]:
name = "Decision Tree"
print("search space:", specs[name]["grid"])
searches[name] = tune_model(name, specs[name], X_train, y_train, cv, N_ITER)

### Model 3: Random Forest

200 trees trained on random samples and averaged, with balanced_subsample class weights.

In [ ]:
name = "Random Forest"
print("search space:", specs[name]["grid"])
searches[name] = tune_model(name, specs[name], X_train, y_train, cv, N_ITER)

### Model 4: Linear SVM

`LinearSVC` finds the straight boundary with the widest margin. It gives margin scores, not probabilities. Uses RobustScaler and clipping.

In [ ]:
name = "Linear SVM"
print("search space:", specs[name]["grid"])
searches[name] = tune_model(name, specs[name], X_train, y_train, cv, N_ITER)

### Model 5: Gradient Boosting

`HistGradientBoostingClassifier`: trees built one after another, each correcting the mistakes of the previous ones.

In [ ]:
name = "Gradient Boosting"
print("search space:", specs[name]["grid"])
searches[name] = tune_model(name, specs[name], X_train, y_train, cv, N_ITER)

### Collect the tuned models

Each search has refit its best candidate on the full training set.

In [ ]:
save_hyperparameters(searches, specs)
models = {n: s.best_estimator_ for n, s in searches.items()}

## Steps 8-9: Evaluation on the untouched test set and comparison

In [ ]:
test_df, scores = evaluate_models(models, X_test, y_test)
comparison = build_comparison(test_df, searches)
comparison.round(4)

## Step 10: Interpretation

In [ ]:
best_name = comparison.index[0]
interpret_models(models, best_name, X_test, y_test)

## Steps 11-12: Save the final model and predict

In [ ]:
save_final_model(best_name, models, searches, comparison, X_train)
save_sample_input(X_test, y_test)
model, meta = load_artifacts()
pred, proba = predict_frame(X_test.head(3), model, meta)
for i, p in enumerate(pred):
    print(format_prediction(p, None if proba is None else proba[i], meta, row=i))